# Big Data Processing and Analysis Project

## Student Information:
> **Full Name**: Vasileios Koutsovasilis

> **Registration Number**: 2020030137  


# PySpark and Ngrok Setup

Steps to setup Apache Spark's Pyspark and Ngrok

In [ ]:
!java -version

openjdk version "11.0.27" 2025-04-15
OpenJDK Runtime Environment (build 11.0.27+6-post-Ubuntu-0ubuntu122.04)
OpenJDK 64-Bit Server VM (build 11.0.27+6-post-Ubuntu-0ubuntu122.04, mixed mode, sharing)


In [ ]:
!curl -Lo coursier https://git.io/coursier-cli

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 42577  100 42577    0     0  29194      0  0:00:01  0:00:01 --:--:--  161k


In [ ]:
!chmod +x coursier

In [ ]:
!./coursier launch --fork almond -- --install

Η έξοδος ροής περικόπηκε στις τελευταίες 5000 γραμμές.
https://repo1.maven.org/maven2/org/fusesource/jansi/jansi/2.4.1/jansi-2.4.1.pom
    0.0% [          ] 0B (0B / s)
https://repo1.maven.org/maven2/org/fusesource/jansi/jansi/2.4.1/jansi-2.4.1.pom
  100.0% [##########] 16.9 KiB (583.5 KiB / s)
https://repo1.maven.org/maven2/org/scalameta/mtags_2.13.16/1.5.1/mtags_2.13.16-…
  100.0% [##########] 4.9 KiB (44.0 KiB / s)
https://repo1.maven.org/maven2/sh/almond/toree-hooks_2.13/0.14.1/toree-hooks_2.…
  100.0% [##########] 1.7 KiB (15.7 KiB / s)
https://repo1.maven.org/maven2/sh/almond/interpreter_2.13/0.14.1/interpreter_2.…
    0.0% [          ] 0B (0B / s)
https://repo1.maven.org/maven2/com/github/alexarchambault/case-app-annotations_…
    0.0% [          ] 0B (0B / s)
https://repo1.maven.org/maven2/sh/almond/interpreter_2.13/0.14.1/interpreter_2.…
    0.0% [          ] 0B (0B / s)
https://repo1.maven.org/maven2/com/github/alexarchambault/case-app-annotations_…
    0.0% [          ] 0B (

In [ ]:
rm -f coursier

In [ ]:
%%capture
!pip install pyspark
!pip install findspark
!pip install pyngrok

In [ ]:
import findspark

findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("Chicago Crime Analytics") \
        .getOrCreate()

spark.conf.set("spark.default.parallelism", 4)

🔔 If you wish to monitor Spark's progress using the GUI visualization:

 ✅**Please create an Ngrok account and use the authtoken provided from the site.**


In [ ]:
from pyngrok import ngrok, conf
import getpass

print("Enter your authtoken, which can be copied "
"from https://dashboard.ngrok.com/get-started/your-authtoken")
conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")



Enter your authtoken, which can be copied from https://dashboard.ngrok.com/get-started/your-authtoken
··········


 * ngrok tunnel "https://<your-tunnel>.ngrok-free.app" -> "http://127.0.0.1:4040"


# Libraries

Necessary Libraries and Dependencies for the project

In [ ]:
from pyspark import SparkFiles
from pyspark.sql.functions import col, hour, dayofweek, month, year, date_format, round, countDistinct, minute
from pyspark.sql.functions import col, to_timestamp, count, to_str, when, sum, regexp_replace, dayofmonth
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampNTZType, DoubleType, NumericType, BooleanType
import random
import time
from pyspark.sql.functions import log1p, expm1

#Loading The Data

Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

🔔 Provide the file path for the Crimes csv file in your Google Drive

In [ ]:
file_path = input("Please Enter the file path: ")

spark.sparkContext.addFile(file_path)

file_name = input("Please Enter the file name (e.g. myFile.csv): ")
df = spark.read.csv(SparkFiles.get(file_name), header=True)
df.printSchema()

df.show(5)

Please Enter the file path: /content/drive/MyDrive/BigData-Datafolder/Crimes_-_2001_to_Present_20250415.csv
root
 |-- ID: string (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: string (nullable = true)
 |-- Domestic: string (nullable = true)
 |-- Beat: string (nullable = true)
 |-- District: string (nullable = true)
 |-- Ward: string (nullable = true)
 |-- Community Area: string (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: string (nullable = true)
 |-- Y Coordinate: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)
 |-- Location: string (nullable = true)

Providing the Proper Schema to the data and converting dates to the proper format.

In [ ]:
schema = StructType([
    StructField("ID", IntegerType()),
    StructField("Case Number", StringType()),
    StructField("Date", StringType()),
    StructField("Block", StringType()),
    StructField("IUCR", StringType()),
    StructField("Primary Type", StringType()),
    StructField("Description", StringType()),
    StructField("Location Description", StringType()),
    StructField("Arrest", BooleanType()),
    StructField("Domestic", BooleanType()),
    StructField("Beat", StringType()),
    StructField("District", StringType()),
    StructField("Ward", IntegerType()),
    StructField("Community Area", StringType()),
    StructField("FBI Code", StringType()),
    StructField("X Coordinate", DoubleType()),
    StructField("Y Coordinate", DoubleType()),
    StructField("Year", IntegerType()),
    StructField("Updated On", StringType()),
    StructField("Latitude", DoubleType()),
    StructField("Longitude", DoubleType()),
    StructField("Location", StringType())
])

df = spark.read.schema(schema).csv(SparkFiles.get("Crimes_-_2001_to_Present_20250415.csv"), header=True)\
          .withColumn("Date", to_timestamp("Date", "MM/dd/yyyy hh:mm:ss a"))\
          .withColumn("Updated On", to_timestamp("Updated On", "MM/dd/yyyy hh:mm:ss a"))

df.show(5)

+--------+-----------+-------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------------------+
|      ID|Case Number|               Date|               Block|IUCR|        Primary Type|         Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|         Updated On|    Latitude|    Longitude|            Location|
+--------+-----------+-------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------------------+
|13311263|   JG503434|2022-07-29 03:39:00|     023XX S TROY ST|1582|OFFENSE INVOLVING...|   CHILD PORNOGRAPHY|           RESIDENCE

# Creating 20% Synopsis using Reservoir Sampling

Defining the function used to create the reservoir sample

In [ ]:
def reservoir_sampling(iterator, sample_size):
  """
  Method to use to create a reservoir sample on each partition.
  """
  reservoir = []
  for i, value in enumerate(iterator):
    if i < sample_size:
      reservoir.append(value)
    else:
      random_num = random.randint(0, i)
      if random_num < sample_size:
        reservoir[random_num] = value

  yield reservoir

In [ ]:
# get the total number of rows to calculate the rows of the sample
total_rows = df.count()
perc = 0.2 # 20% sample
sample_rows = int(total_rows*perc)
oversample_rate = 1

# cast date and updated on columns to strings to preserve the date data for the transformations
rdd = df.withColumn("Date", col("Date").cast("string"))\
       .withColumn("Updated On", col("Updated On").cast("string"))\
       .rdd

# Get the number of partitions to calculate how many samples to take per partition
num_of_partitions = int(rdd.getNumPartitions())
sample_per_partition = int(oversample_rate*sample_rows/num_of_partitions)
sample_num_of_partitions = int(num_of_partitions*perc)

# create the synopsis rdd
sampled_rdd = rdd.mapPartitions(lambda x: reservoir_sampling(x, sample_per_partition))\
                 .flatMap(lambda x: x)\
                 .mapPartitions(lambda x: reservoir_sampling(x, sample_rows))\
                 .flatMap(lambda x: x)

# make the rdd into a dataframe, repartition it to have the same amount of partitions
# as the initial dataset, make it persistant so that it isn't calculated each time
# and recast the Date and Updated On columns into date formats
synopsis_df = spark.createDataFrame(sampled_rdd, schema)\
                   .repartition(sample_num_of_partitions)\
                   .persist()\
                   .withColumn("Date", to_timestamp("Date"))\
                   .withColumn("Updated On", to_timestamp("Updated On"))

# show some rows of the final synopsis
synopsis_df.show()

total_sampled_rows = synopsis_df.count()
print(f"Total rows of final df: {int(total_sampled_rows)}")

+--------+-----------+-------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------------------+
|      ID|Case Number|               Date|               Block|IUCR|        Primary Type|         Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|         Updated On|    Latitude|    Longitude|            Location|
+--------+-----------+-------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------------------+
|13260322|   JG483918|2023-10-29 23:53:00|   076XX S DANTE AVE|0910| MOTOR VEHICLE THEFT|          AUTOMOBILE|              STREET

# Queries

Queries to showcase the correctness of the synopsis creation methodology

## Days Ranked By Crime Percentage

In [ ]:
start_og = time.time()
total_crimes = int(df.count())

days_ranked = df.select(col("ID"), date_format("Date", "EEEE").alias("Day of the week"))\
                .groupBy("Day of the week")\
                .count()\
                .withColumn("Crime percentage", round((col("count")/total_crimes)*100, 2))\
                .orderBy("Crime percentage", ascending=False)\
                .select(col("Day of the week"), col("Crime percentage"))

print("Days Ranked By Crime Percentage (Original Dataset):")
days_ranked.show()
end_og = time.time()

start_syn = time.time()
total_crimes_syn = int(synopsis_df.count())

days_ranked_syn = synopsis_df.select(col("ID"), date_format("Date", "EEEE").alias("Day of the week"))\
                             .groupBy("Day of the week")\
                             .count()\
                             .withColumn("Crime percentage", round((col("count")/total_crimes_syn)*100, 2))\
                             .orderBy("Crime percentage", ascending=False)\
                             .select(col("Day of the week"), col("Crime percentage"))

print("Days Ranked By Crime Percentage (Synopsis):")
days_ranked_syn.show()
end_syn = time.time()

print(f"Query on original DataFrame took: {end_og-start_og} sec.")
print(f"Query on synopsis DataFrame took: {end_syn-start_syn} sec.")

Days Ranked By Crime Percentage (Original Dataset):
+---------------+----------------+
|Day of the week|Crime percentage|
+---------------+----------------+
|         Friday|            15.0|
|      Wednesday|           14.37|
|       Saturday|           14.34|
|        Tuesday|           14.28|
|       Thursday|           14.22|
|         Monday|           14.14|
|         Sunday|           13.65|
+---------------+----------------+

Days Ranked By Crime Percentage (Synopsis):
+---------------+----------------+
|Day of the week|Crime percentage|
+---------------+----------------+
|         Friday|           14.95|
|      Wednesday|           14.38|
|       Saturday|           14.31|
|        Tuesday|           14.29|
|       Thursday|           14.26|
|         Monday|           14.15|
|         Sunday|           13.65|
+---------------+----------------+

Query on original DataFrame took: 58.521920919418335 sec.
Query on synopsis DataFrame took: 2.503554582595825 sec.


## Top 5 months with the most crimes

In [ ]:
top_5_months = df.groupBy(date_format("Date", "MMMM").alias("Month"))\
                 .count()\
                 .withColumnRenamed("count", "crimes")\
                 .orderBy("crimes", ascending=False)\
                 .limit(5)

print("Top-5 months with most crimes (Original DF):")
start_og = time.time()
top_5_months.show()
end_og = time.time()

sampled_top_5_months = synopsis_df.groupBy(date_format("Date", "MMMM").alias("Month"))\
                                  .count()\
                                  .withColumnRenamed("count", "crimes")\
                                  .orderBy("crimes", ascending=False)\
                                  .withColumn("crimes", col("crimes")*5)\
                                  .limit(5)

print("Top-5 months with most crimes (Synopsis):")
start_syn = time.time()
sampled_top_5_months.show()
end_syn = time.time()

print(f"Query on original DataFrame took: {end_og-start_og} sec.")
print(f"Query on synopsis DataFrame took: {end_syn-start_syn} sec.")

Top-5 months with most crimes (Original DF):
+-------+------+
|  Month|crimes|
+-------+------+
|   July|765501|
| August|757798|
|    May|728340|
|   June|727809|
|October|721938|
+-------+------+

Top-5 months with most crimes (Synopsis):
+-------+------+
|  Month|crimes|
+-------+------+
|   July|766580|
| August|756150|
|    May|731060|
|   June|728040|
|October|716550|
+-------+------+

Query on original DataFrame took: 47.7045795917511 sec.
Query on synopsis DataFrame took: 2.356328010559082 sec.


## Arrest Percentages of Domestic and Non Domestic Cases

In [ ]:
perc_of_arrests = df.select(col("Domestic"), col("Arrest"))\
                    .rdd\
                    .keyBy(lambda x: x['Domestic'])\
                    .mapValues(lambda x: (1, 1 if x['Arrest'] == True else 0))\
                    .reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1]))\
                    .mapValues(lambda x: x[1]/x[0]*100)



print("Percentage of Arrests(Original Dataset)")
start_og = time.time()
for key,value in perc_of_arrests.collect():
  print(f"Domestic Case: {key} || Arrest Percentage: {value:.3f}%")
end_og = time.time()
print("===============================================")

synopsis_perc_of_arrests = synopsis_df.select(col("Domestic"), col("Arrest"))\
                                      .rdd\
                                      .keyBy(lambda x: x['Domestic'])\
                                      .mapValues(lambda x: (1, 1 if x['Arrest'] == True else 0))\
                                      .reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1]))\
                                      .mapValues(lambda x: x[1]/x[0]*100)

print("Percentage of Arrests(Synopsis):")
start_syn = time.time()
for key,value in perc_of_arrests.collect():
  print(f"Domestic Case: {key} || Arrest Percentage: {value:.3f}%")
end_syn = time.time()

print(f"\n\nQuery on original DataSet took: {end_og-start_og} sec.")
print(f"Query on Synopsis took: {end_syn-start_syn} sec.")


Percentage of Arrests(Original Dataset)
Domestic Case: False || Arrest Percentage: 26.646%
Domestic Case: True || Arrest Percentage: 19.301%
Percentage of Arrests(Synopsis):
Domestic Case: False || Arrest Percentage: 26.646%
Domestic Case: True || Arrest Percentage: 19.301%


Query on original DataSet took: 85.26183915138245 sec.
Query on Synopsis took: 1.9232797622680664 sec.


# Crime Count Predictor using Machine Learning

Steps for creating the Spatiotemporal Crime Count Predictor utilizing the full Dataset

## Data Cleaning

Find the total `Null` values for every Column.

In [ ]:
null_vals = df.select(*(sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns))
null_vals.show()

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
| ID|Case Number|Date|Block|IUCR|Primary Type|Description|Location Description|Arrest|Domestic|Beat|District|  Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|  0|          0|   0|    0|   0|           0|          0|               14184|     0|       0|   0|      47|614825|        613452|       0|       91870|       91870|   0|         0|   91870|    91870|   91870|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+---

Since we want to make a District level predictor we need to fill the `Null` District entries. The Beat column has no `Null` values, so for the `Null` District entries we find the most frequent Beat.

In [ ]:
df.filter(col("District").isNull())\
  .select(col("Beat"))\
  .summary()\
  .show()

+-------+------------------+
|summary|              Beat|
+-------+------------------+
|  count|                47|
|   mean|1357.6382978723404|
| stddev|  748.192762465486|
|    min|              0124|
|    25%|             731.0|
|    50%|            1333.0|
|    75%|            1912.0|
|    max|              2535|
+-------+------------------+



Afterwards use the most frequent Beat to find the most frequent District for that Beat.

In [ ]:
df.filter(col("Beat") == 1912)\
  .select(col("District"))\
  .summary()\
  .show()

+-------+--------------------+
|summary|            District|
+-------+--------------------+
|  count|               18949|
|   mean|   18.99989445353317|
| stddev|0.014529037602816207|
|    min|                 017|
|    25%|                19.0|
|    50%|                19.0|
|    75%|                19.0|
|    max|                 019|
+-------+--------------------+



Finally fill the `Null` District entries with the most frequent District value.

In [ ]:
no_null_district_df = df.fillna({"District": "019"})

null_vals = no_null_district_df.select(*(sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns))
null_vals.show()

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
| ID|Case Number|Date|Block|IUCR|Primary Type|Description|Location Description|Arrest|Domestic|Beat|District|  Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|  0|          0|   0|    0|   0|           0|          0|               14184|     0|       0|   0|       0|614825|        613452|       0|       91870|       91870|   0|         0|   91870|    91870|   91870|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+---

In [ ]:
rare_crimes_threshold = 4000
rare_crimes = no_null_district_df.groupBy("Primary Type")\
                                 .count()\
                                 .orderBy("count", ascending=False)\
                                 .filter(col("count") < rare_crimes_threshold)\
                                 .select(col("Primary Type"))\
                                 .collect()

rare_crimes_list = [row[0] for row in rare_crimes]

reduced_ptype_cardinality_dataset = no_null_district_df.withColumn("Primary Type", when(col("Primary Type").isin(rare_crimes_list), "Rare Crime")
                                                                             .otherwise(col("Primary Type")))


In [ ]:
print(reduced_ptype_cardinality_dataset.count())

8294214


## Preprocessing

In [ ]:
processed_dataset = reduced_ptype_cardinality_dataset.withColumn("Day", dayofweek(col("Date")))\
                                                     .withColumn("Month", month(col("Date")))\
                                                     .withColumn("District", col("District").cast("int"))

processed_dataset.show()

+--------+-----------+-------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------------------+---+-----+
|      ID|Case Number|               Date|               Block|IUCR|        Primary Type|         Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|         Updated On|    Latitude|    Longitude|            Location|Day|Month|
+--------+-----------+-------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------------------+---+-----+
|13311263|   JG503434|2022-07-29 03:39:00|     023XX S TROY ST|1582|OFFENSE INVOLVING...|   CHILD PO

## Feature Selection and Training

In [ ]:
dataset_for_ml = processed_dataset.groupBy("Primary Type", "Year", "Month", "Day", "District")\
                                  .count()\
                                  .withColumnRenamed("count", "Crime Count")\
                                  .withColumn("log Count", log1p("Crime Count"))\
                                  .orderBy("Year")

dataset_for_ml.show()

+-------------------+----+-----+---+--------+-----------+------------------+
|       Primary Type|Year|Month|Day|District|Crime Count|         log Count|
+-------------------+----+-----+---+--------+-----------+------------------+
| DECEPTIVE PRACTICE|2001|    3|  4|       1|         27| 3.332204510175204|
|MOTOR VEHICLE THEFT|2001|    1|  2|       2|         14|  2.70805020110221|
|         Rare Crime|2001|    3|  2|      14|          1|0.6931471805599453|
|              THEFT|2001|    5|  7|      24|         32|3.4965075614664802|
|            ASSAULT|2001|    2|  5|      24|         11|2.4849066497880004|
|      OTHER OFFENSE|2001|    1|  6|       8|         16| 2.833213344056216|
|          NARCOTICS|2001|    3|  2|      18|         12|2.5649493574615367|
|            ROBBERY|2001|    3|  1|       6|         12|2.5649493574615367|
|      OTHER OFFENSE|2001|    3|  5|       5|         25| 3.258096538021482|
|            ROBBERY|2001|    4|  4|       8|         12|2.5649493574615367|

In [ ]:
print("Total Rows:", dataset_for_ml.count())

Total Rows: 713293


In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.pipeline import Pipeline

feature_columns = ["PType Vector", "District Vector", "Year", "Month", "Day"]

ptype_indexer = StringIndexer(inputCol="Primary Type", outputCol="PType Indexed", handleInvalid="keep")
district_indexer = StringIndexer(inputCol="District", outputCol="District Indexed", handleInvalid="keep")
ptype_encoder = OneHotEncoder(inputCol="PType Indexed", outputCol="PType Vector")
district_encoder = OneHotEncoder(inputCol="District Indexed", outputCol="District Vector")

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

df_ptype_indexed = ptype_indexer.fit(dataset_for_ml).transform(dataset_for_ml)
df_district_indexed = district_indexer.fit(df_ptype_indexed).transform(df_ptype_indexed)

df_ptype_encoded = ptype_encoder.fit(df_district_indexed).transform(df_district_indexed)
df_fully_encoded = district_encoder.fit(df_ptype_encoded).transform(df_ptype_encoded)

df_assembled = assembler.transform(df_fully_encoded)

columns_to_keep = ["features", "log Count", "Crime Count"]+feature_columns
columns_to_drop = [c for c in df_assembled.columns if c not in columns_to_keep]
df_assembled_simplified = df_assembled.drop(*columns_to_drop)

train_data = df_assembled_simplified.filter(col("Year") <= 2020)
test_data = df_assembled_simplified.filter(col("Year").between(2021, 2024))

In [ ]:
gbt = GBTRegressor(featuresCol="features", labelCol="log Count", maxIter=100)
start_og = time.time()
gbt_model = gbt.fit(train_data)
end_og = time.time()

elapsed_time = end_og-start_og

print("Gradient Boosted Trees Regression:")
print(f"Training took: {elapsed_time:.2f} sec.")
gbt_predictions = gbt_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))
gbt_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Gradient Boosted Trees Regression:
Training took: 537.47 sec.
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...| 2.553018036387756|2.9444389791664403|            11.8|         18|
|(54,[5,29,51,52,5...| 2.105417694580811|1.9459101490553132|             7.2|          6|
|(54,[5,38,51,52,5...|1.5808398778688992|2.1972245773362196|             3.9|          8|
|(54,[10,28,51,52,...|2.9649225190171804|3.4965075614664802|            18.4|         32|
|(54,[0,39,51,52,5...|3.1440779502101495|3.6635616461296463|            22.2|         38|
|(54,[0,35,51,52,5...|3.2042438000211098| 3.044522437723423|            23.6|         20|
|(54,[5,42,51,52,5...|2.3787175408744607|2.4849066497880004|             9.8|         11|
|(54,[4,39,51,52,5...|2.10810550502362

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(featuresCol="features", labelCol="log Count", maxDepth=20)
start_og = time.time()
dt_model = dt.fit(train_data)
end_og = time.time()

elapsed_time = end_og-start_og
print("Decision Tree:")
print(f"Training took: {elapsed_time:.2f} sec.")
dt_predictions = dt_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))
dt_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Decision Tree:
Training took: 125.71 sec.
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...|  3.49650756146648|2.9444389791664403|            32.0|         18|
|(54,[5,29,51,52,5...|2.2688082088681996|1.9459101490553132|             8.7|          6|
|(54,[5,38,51,52,5...|1.5672738439037128|2.1972245773362196|             3.8|          8|
|(54,[10,28,51,52,...|3.2188758248682006|3.4965075614664802|            24.0|         32|
|(54,[0,39,51,52,5...| 3.076396594596972|3.6635616461296463|            20.7|         38|
|(54,[0,35,51,52,5...| 3.076396594596972| 3.044522437723423|            20.7|         20|
|(54,[5,42,51,52,5...|1.3862943611198906|2.4849066497880004|             3.0|         11|
|(54,[4,39,51,52,5...| 1.999182774737431|2.397895272798370

In [ ]:
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol="features", labelCol="log Count", maxIter=100)

start_og = time.time()
lr_model = lr.fit(train_data)
end_og = time.time()

elapsed_time = end_og - start_og
print("Linear Regression:")
print(f"Training took: {elapsed_time:.2f} sec.")
lr_predictions = lr_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))
lr_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Linear Regression:
Training took: 157.36 sec.
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...|1.6641222600437615|2.9444389791664403|             4.3|         18|
|(54,[5,29,51,52,5...| 1.975442448792542|1.9459101490553132|             6.2|          6|
|(54,[5,38,51,52,5...|1.6941740832326246|2.1972245773362196|             4.4|          8|
|(54,[10,28,51,52,...| 2.576390490346391|3.4965075614664802|            12.1|         32|
|(54,[0,39,51,52,5...| 3.378623006980405|3.6635616461296463|            28.3|         38|
|(54,[0,35,51,52,5...| 3.472235414242668| 3.044522437723423|            31.2|         20|
|(54,[5,42,51,52,5...|1.7683792803467924|2.4849066497880004|             4.9|         11|
|(54,[4,39,51,52,5...| 2.207698656549283|2.39789527279

## Regression Evaluation

In [ ]:
def print_metrics(predictions, algorithm: str, label_name: str):
  metrics = ["r2", "rmse", "mae"]
  print("====================================================")
  print(f"Evaluation of {algorithm}: ")
  for metric in metrics:
    re = RegressionEvaluator(labelCol=label_name, predictionCol="prediction", metricName=metric)
    eval = re.evaluate(predictions)
    print(f"{metric}: {eval}")
  print("====================================================\n")

In [ ]:
print_metrics(gbt_predictions, "GBT Regressor", "log Count")
print_metrics(lr_predictions, "Linear Regression", "log Count")
print_metrics(dt_predictions, "Decision Tree", "log Count")

Evaluation of GBT Regressor: 
r2: 0.7446605106593917
rmse: 0.4657800142375841
mae: 0.3566983207807855

Evaluation of Linear Regression: 
r2: 0.5499149275398743
rmse: 0.6183995455430762
mae: 0.47781038017994243

Evaluation of Decision Tree: 
r2: 0.7151804733695188
rmse: 0.49193388376694175
mae: 0.37332932119008516



# Creating Synopses for Training using Reservoir Sampling

Steps for creating the different synopses on which the ml model will be trained

In [ ]:
train_data_to_sample = reduced_ptype_cardinality_dataset.filter(col("Year") <= 2020)
total_training_rows = train_data_to_sample.count()
training_rdd = train_data_to_sample.withColumn("Date", col("Date").cast("string")).rdd
num_of_partitions = training_rdd.getNumPartitions()

## 40% Sample Synopsis Creation

In [ ]:
perc_40 = 0.4
sample_rows = int(perc_40*total_training_rows)

oversample_rate = 1.144
sample_per_partition = int(sample_rows*oversample_rate/num_of_partitions)
sample_40_perc_partitions = int(num_of_partitions*perc_40)

perc_40_sample_rdd = training_rdd.mapPartitions(lambda x: reservoir_sampling(x, sample_per_partition))\
                                 .flatMap(lambda x: x)\
                                 .mapPartitions(lambda x: reservoir_sampling(x, sample_rows))\
                                 .flatMap(lambda x: x)


synopsis_40_perc = spark.createDataFrame(perc_40_sample_rdd, schema)\
                                 .repartition(sample_40_perc_partitions)\
                                 .persist()\
                                 .withColumn("Date", to_timestamp("Date"))\
                                 .withColumn("Day", dayofweek(col("Date")))\
                                 .withColumn("Month", month(col("Date")))\
                                 .withColumn("District", col("District").cast("int"))

print("40% synopsis rows: ", synopsis_40_perc.count(), f" vs original training rows: {total_training_rows}.")

training_synopsis_40_perc = synopsis_40_perc.groupBy("Primary Type", "Year", "Month", "Day", "District")\
                                            .count()\
                                            .withColumnRenamed("count", "Crime Count")\
                                            .withColumn("Crime Count", col("Crime Count")*2.5)\
                                            .withColumn("log Count", log1p("Crime Count"))\
                                            .orderBy("Year")

training_synopsis_40_perc.show()

40% synopsis rows:  2907074  vs original training rows: 7267103.
+-------------------+----+-----+---+--------+-----------+------------------+
|       Primary Type|Year|Month|Day|District|Crime Count|         log Count|
+-------------------+----+-----+---+--------+-----------+------------------+
|            ASSAULT|2001|    5|  1|      20|       15.0| 2.772588722239781|
| DECEPTIVE PRACTICE|2001|    3|  4|       1|       32.5|3.5115454388310208|
|MOTOR VEHICLE THEFT|2001|    6|  1|      22|       17.5| 2.917770732084279|
|CRIM SEXUAL ASSAULT|2001|    8|  3|      16|        2.5| 1.252762968495368|
|      OTHER OFFENSE|2001|    1|  6|       8|       15.0| 2.772588722239781|
|            BATTERY|2001|    5|  5|      19|       50.0|3.9318256327243257|
|          NARCOTICS|2001|    3|  4|      15|       70.0|4.2626798770413155|
|  CRIMINAL TRESPASS|2001|    4|  5|       9|       20.0| 3.044522437723423|
|            BATTERY|2001|    8|  3|      12|       40.0| 3.713572066704308|
|          

## 30% Sample Synopsis Creation

In [ ]:
perc_30 = 0.3
sample_rows = int(perc_30*total_training_rows)

oversample_rate = 1.141
sample_per_partition = int(sample_rows*oversample_rate/num_of_partitions)
sample_30_perc_partitions = int(num_of_partitions*perc_30)

perc_30_sample_rdd = training_rdd.mapPartitions(lambda x: reservoir_sampling(x, sample_per_partition))\
                                 .flatMap(lambda x: x)\
                                 .mapPartitions(lambda x: reservoir_sampling(x, sample_rows))\
                                 .flatMap(lambda x: x)

synopsis_30_perc = spark.createDataFrame(perc_30_sample_rdd, schema)\
                                 .repartition(sample_30_perc_partitions)\
                                 .persist()\
                                 .withColumn("Date", to_timestamp("Date"))\
                                 .withColumn("Day", dayofweek(col("Date")))\
                                 .withColumn("Month", month(col("Date")))\
                                 .withColumn("District", col("District").cast("int"))

print("30% synopsis rows: ", synopsis_30_perc.count(), f" vs original training rows: {total_training_rows}.")

training_synopsis_30_perc = synopsis_30_perc.groupBy("Primary Type", "Year", "Month", "Day", "District")\
                                            .count()\
                                            .withColumnRenamed("count", "Crime Count")\
                                            .withColumn("Crime Count", col("Crime Count")*3.333)\
                                            .withColumn("log Count", log1p("Crime Count"))\
                                            .orderBy("Year")

training_synopsis_30_perc.show()

30% synopsis rows:  2180894  vs original training rows: 7267103.
+------------------+----+-----+---+--------+------------------+------------------+
|      Primary Type|Year|Month|Day|District|       Crime Count|         log Count|
+------------------+----+-----+---+--------+------------------+------------------+
|      PROSTITUTION|2001|    4|  6|      25|            29.997|3.4338904256086735|
|DECEPTIVE PRACTICE|2001|    1|  7|      19|            16.665|2.8715852808111086|
|      PROSTITUTION|2001|    5|  5|      25|             6.666|2.0367949669583636|
|          BURGLARY|2001|    9|  7|       5|23.331000000000003| 3.191751257471543|
|           BATTERY|2001|    2|  7|      18|            29.997|3.4338904256086735|
|           ASSAULT|2001|    3|  2|       3|            26.664| 3.320131928940935|
|           BATTERY|2001|    7|  1|      19|             66.66| 4.214495163352562|
|           BATTERY|2001|    7|  1|      12|103.32300000000001| 4.647491855433561|
|           BATTERY|20

## 20% Sample Synopsis Creation

In [ ]:
perc_20 = 0.2
sample_rows = int(perc_20*total_training_rows)

oversample_rate = 1.14
sample_per_partition = int(sample_rows*oversample_rate/num_of_partitions)
sample_20_perc_partitions = int(num_of_partitions*perc_20)

perc_20_sample_rdd = training_rdd.mapPartitions(lambda x: reservoir_sampling(x, sample_per_partition))\
                                 .flatMap(lambda x: x)\
                                 .mapPartitions(lambda x: reservoir_sampling(x, sample_rows))\
                                 .flatMap(lambda x: x)


synopsis_20_perc = spark.createDataFrame(perc_20_sample_rdd, schema)\
                                 .repartition(sample_20_perc_partitions)\
                                 .persist()\
                                 .withColumn("Date", to_timestamp("Date"))\
                                 .withColumn("Day", dayofweek(col("Date")))\
                                 .withColumn("Month", month(col("Date")))\
                                 .withColumn("District", col("District").cast("int"))

print("20% synopsis rows: ", synopsis_20_perc.count(), f" vs original training rows: {total_training_rows}.")

training_synopsis_20_perc = synopsis_20_perc.groupBy("Primary Type", "Year", "Month", "Day", "District")\
                                            .count()\
                                            .withColumnRenamed("count", "Crime Count")\
                                            .withColumn("Crime Count", col("Crime Count")*5)\
                                            .withColumn("log Count", log1p("Crime Count"))\
                                            .orderBy("Year")

training_synopsis_40_perc.show()

20% synopsis rows:  1461006  vs original training rows: 7267103.
+-------------------+----+-----+---+--------+-----------+------------------+
|       Primary Type|Year|Month|Day|District|Crime Count|         log Count|
+-------------------+----+-----+---+--------+-----------+------------------+
| DECEPTIVE PRACTICE|2001|    3|  4|       1|       32.5|3.5115454388310208|
|            ASSAULT|2001|    5|  1|      20|       15.0| 2.772588722239781|
|CRIM SEXUAL ASSAULT|2001|    8|  3|      16|        2.5| 1.252762968495368|
|MOTOR VEHICLE THEFT|2001|    6|  1|      22|       17.5| 2.917770732084279|
|            BATTERY|2001|    5|  5|      19|       50.0|3.9318256327243257|
|      OTHER OFFENSE|2001|    1|  6|       8|       15.0| 2.772588722239781|
|  CRIMINAL TRESPASS|2001|    4|  5|       9|       20.0| 3.044522437723423|
|          NARCOTICS|2001|    3|  4|      15|       70.0|4.2626798770413155|
|            BATTERY|2001|    4|  6|       7|       77.5| 4.363098624788362|
|          

# Training on the Synopses

Training the ml models on the synopses of various sizes

## Training on 40% Synopsis

In [ ]:
syn_40_ptype_indexed = ptype_indexer.fit(training_synopsis_40_perc).transform(training_synopsis_40_perc)
syn_40_district_indexed = district_indexer.fit(syn_40_ptype_indexed).transform(syn_40_ptype_indexed)

syn_40_ptype_encoded = ptype_encoder.fit(syn_40_district_indexed).transform(syn_40_district_indexed)
syn_40_fully_encoded = district_encoder.fit(syn_40_ptype_encoded).transform(syn_40_ptype_encoded)

syn_40_assembled = assembler.transform(syn_40_fully_encoded)

syn_40_training = syn_40_assembled.drop(*columns_to_drop)

In [ ]:
gbt = GBTRegressor(featuresCol="features", labelCol="log Count", maxIter=100)

start_syn = time.time()
gbt_syn_40_model = gbt.fit(syn_40_training)
end_syn = time.time()

gbt_syn_40_predictions = gbt_syn_40_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))

elapsed_time = end_syn-start_syn
print(f"Training GBTR on 40% synopsis took: {elapsed_time:.2f} sec.")

print("Gradient Boosted Trees (Training on 40% synopsis):")
gbt_syn_40_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Training GBTR on 40% synopsis took: 419.73 sec.
Gradient Boosted Trees (Training on 40% synopsis):
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...|1.4674948424519658|2.9444389791664403|             3.3|         18|
|(54,[5,29,51,52,5...|  2.01803928459268|1.9459101490553132|             6.5|          6|
|(54,[5,38,51,52,5...|1.7580607805343345|2.1972245773362196|             4.8|          8|
|(54,[10,28,51,52,...|1.8834227061382673|3.4965075614664802|             5.6|         32|
|(54,[0,39,51,52,5...|  3.23224034005508|3.6635616461296463|            24.3|         38|
|(54,[0,35,51,52,5...|3.1902909969040136| 3.044522437723423|            23.3|         20|
|(54,[5,42,51,52,5...|1.8134087739301075|2.4849066497880004|             5.1|         11|
|

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(featuresCol="features", labelCol="log Count", maxDepth=20)

start_syn = time.time()
dt_syn_40_model = dt.fit(syn_40_training)
end_syn = time.time()

dt_syn_40_predictions = dt_syn_40_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))

elapsed_time = end_syn-start_syn
print(f"Training DTR on 40% synopsis took: {elapsed_time:.2f} sec.")
print("Decision Tree (Training on 40% Synopsis):")
dt_syn_40_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Training DTR on 40% synopsis took: 73.30 sec.
Decision Tree (Training on 40% Synopsis):
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...| 1.486382151655405|2.9444389791664403|             3.4|         18|
|(54,[5,29,51,52,5...|1.6406662191167047|1.9459101490553132|             4.2|          6|
|(54,[5,38,51,52,5...|1.6406662191167047|2.1972245773362196|             4.2|          8|
|(54,[10,28,51,52,...|1.5089706224540855|3.4965075614664802|             3.5|         32|
|(54,[0,39,51,52,5...| 2.917770732084279|3.6635616461296463|            17.5|         38|
|(54,[0,35,51,52,5...|2.9152883324452383| 3.044522437723423|            17.5|         20|
|(54,[5,42,51,52,5...|1.8865816732024132|2.4849066497880004|             5.6|         11|
|(54,[4,39,5

In [ ]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="log Count", maxIter=100)

start_syn = time.time()
lr_syn_40_model = lr.fit(syn_40_training)
end_syn = time.time()

elapsed_time = end_syn - start_syn
print(f"Training Linear Regression on 40% synopsis took: {elapsed_time:.2f} sec.")
print("Linear Regression (Training on 40% Synopsis):")
lr_syn_40_predictions = lr_syn_40_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))
lr_syn_40_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Training Linear Regression on 40% synopsis took: 38.69 sec.
Linear Regression (Training on 40% Synopsis):
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...|1.9616708916051664|2.9444389791664403|             6.1|         18|
|(54,[5,29,51,52,5...| 2.235411642184218|1.9459101490553132|             8.4|          6|
|(54,[5,38,51,52,5...| 2.066450101877656|2.1972245773362196|             6.9|          8|
|(54,[10,28,51,52,...|1.7995565798213775|3.4965075614664802|             5.0|         32|
|(54,[0,39,51,52,5...|3.2758337488072584|3.6635616461296463|            25.5|         38|
|(54,[0,35,51,52,5...| 3.338690628243455| 3.044522437723423|            27.2|         20|
|(54,[5,42,51,52,5...|1.9008141558292735|2.4849066497880004|             5.7|       

## Training on 30% Synopsis

In [ ]:
syn_30_ptype_indexed = ptype_indexer.fit(training_synopsis_30_perc).transform(training_synopsis_30_perc)
syn_30_district_indexed = district_indexer.fit(syn_30_ptype_indexed).transform(syn_30_ptype_indexed)

syn_30_ptype_encoded = ptype_encoder.fit(syn_30_district_indexed).transform(syn_30_district_indexed)
syn_30_fully_encoded = district_encoder.fit(syn_30_ptype_encoded).transform(syn_30_ptype_encoded)

syn_30_assembled = assembler.transform(syn_30_fully_encoded)

syn_30_training = syn_30_assembled.drop(*columns_to_drop)

In [ ]:
gbt = GBTRegressor(featuresCol="features", labelCol="log Count", maxIter=100)

start_syn = time.time()
gbt_syn_30_model = gbt.fit(syn_30_training)
end_syn = time.time()

gbt_syn_30_predictions = gbt_syn_30_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))

elapsed_time = end_syn-start_syn
print(f"Training GBTR on 30% synopsis took: {elapsed_time:.2f} sec.")

print("Gradient Boosted Trees Regression (Training on 30% synopsis):")
gbt_syn_30_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Training GBTR on 30% synopsis took: 362.77 sec.
Gradient Boosted Trees Regression (Training on 30% synopsis):
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...|1.6963755978599466|2.9444389791664403|             4.5|         18|
|(54,[5,29,51,52,5...|2.1645572469878855|1.9459101490553132|             7.7|          6|
|(54,[5,38,51,52,5...| 1.813719639387207|2.1972245773362196|             5.1|          8|
|(54,[10,28,51,52,...|1.9458313109587795|3.4965075614664802|             6.0|         32|
|(54,[0,39,51,52,5...|3.1748006364956236|3.6635616461296463|            22.9|         38|
|(54,[0,35,51,52,5...|3.2396146132398895| 3.044522437723423|            24.5|         20|
|(54,[5,42,51,52,5...| 1.890386122780424|2.4849066497880004|             5.6|   

In [ ]:
dt = DecisionTreeRegressor(featuresCol="features", labelCol="log Count", maxDepth=20)

start_syn = time.time()
dt_syn_30_model = dt.fit(syn_30_training)
end_syn = time.time()

dt_syn_30_predictions = dt_syn_30_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))

elapsed_time = end_syn-start_syn
print(f"Training DTR on 30% synopsis took: {elapsed_time:.2f} sec.")
print("Decision Tree (Training on 30% Synopsis):")
dt_syn_30_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Training DTR on 30% synopsis took: 66.33 sec.
Decision Tree (Training on 30% Synopsis):
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...|1.8755471297826678|2.9444389791664403|             5.5|         18|
|(54,[5,29,51,52,5...|1.6540200228849975|1.9459101490553132|             4.2|          6|
|(54,[5,38,51,52,5...|1.7857805393631203|2.1972245773362196|             5.0|          8|
|(54,[10,28,51,52,...|1.7239210311064264|3.4965075614664802|             4.6|         32|
|(54,[0,39,51,52,5...|2.6624947994427073|3.6635616461296463|            13.3|         38|
|(54,[0,35,51,52,5...| 3.188197964194496| 3.044522437723423|            23.2|         20|
|(54,[5,42,51,52,5...|2.0773643091254295|2.4849066497880004|             7.0|         11|
|(54,[4,39,5

In [ ]:
lr = LinearRegression(featuresCol="features", labelCol="log Count", maxIter=100)

start_syn = time.time()
lr_syn_30_model = lr.fit(syn_30_training)
end_syn = time.time()

elapsed_time = end_syn - start_syn
print(f"Training Linear Regression on 30% synopsis took: {elapsed_time:.2f} sec.")
print("Linear Regression (Training on 30% Synopsis):")

lr_syn_30_predictions = lr_syn_30_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))
lr_syn_30_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Training Linear Regression on 30% synopsis took: 30.58 sec.
Linear Regression (Training on 30% Synopsis):
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...|2.0306139881935863|2.9444389791664403|             6.6|         18|
|(54,[5,29,51,52,5...|2.2949585357839055|1.9459101490553132|             8.9|          6|
|(54,[5,38,51,52,5...| 2.126196685624244|2.1972245773362196|             7.4|          8|
|(54,[10,28,51,52,...| 1.906867394564081|3.4965075614664802|             5.7|         32|
|(54,[0,39,51,52,5...|3.2837994368471826|3.6635616461296463|            25.7|         38|
|(54,[0,35,51,52,5...|3.3368613890373666| 3.044522437723423|            27.1|         20|
|(54,[5,42,51,52,5...|1.9618240900119162|2.4849066497880004|             6.1|       

## Training on 20% Synopsis

In [ ]:
syn_20_ptype_indexed = ptype_indexer.fit(training_synopsis_20_perc).transform(training_synopsis_20_perc)
syn_20_district_indexed = district_indexer.fit(syn_20_ptype_indexed).transform(syn_20_ptype_indexed)

syn_20_ptype_encoded = ptype_encoder.fit(syn_20_district_indexed).transform(syn_20_district_indexed)
syn_20_fully_encoded = district_encoder.fit(syn_20_ptype_encoded).transform(syn_20_ptype_encoded)

syn_20_assembled = assembler.transform(syn_20_fully_encoded)

syn_20_training = syn_20_assembled.drop(*columns_to_drop)

In [ ]:
gbt = GBTRegressor(featuresCol="features", labelCol="log Count", maxIter=100)

start_syn = time.time()
gbt_syn_20_model = gbt.fit(syn_20_training)
end_syn = time.time()

gbt_syn_20_predictions = gbt_syn_20_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))

elapsed_time = end_syn-start_syn
print(f"Training GBTR on 20% synopsis took: {elapsed_time:.2f} sec.")

print("Gradient Boosted Trees Regression (Training on 20% synopsis):")
gbt_syn_20_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

In [ ]:
dt = DecisionTreeRegressor(featuresCol="features", labelCol="log Count", maxDepth=20)

start_syn = time.time()
dt_syn_20_model = dt.fit(syn_20_training)
end_syn = time.time()

dt_syn_20_predictions = dt_syn_20_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))

elapsed_time = end_syn-start_syn
print(f"Training DTR on 20% synopsis took: {elapsed_time:.2f} sec.")
print("Decision Tree (Training on 20% Synopsis):")
dt_syn_20_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Training DTR on 20% synopsis took: 59.06 sec.
Decision Tree (Training on 20% Synopsis):
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...|  1.93198745703862|2.9444389791664403|             5.9|         18|
|(54,[5,29,51,52,5...|1.9938047370848269|1.9459101490553132|             6.3|          6|
|(54,[5,38,51,52,5...|2.0262121292842923|2.1972245773362196|             6.6|          8|
|(54,[10,28,51,52,...|2.0238212734099807|3.4965075614664802|             6.6|         32|
|(54,[0,39,51,52,5...| 2.772588722239781|3.6635616461296463|            15.0|         38|
|(54,[0,35,51,52,5...|3.3415066598340815| 3.044522437723423|            27.3|         20|
|(54,[5,42,51,52,5...|  1.93198745703862|2.4849066497880004|             5.9|         11|
|(54,[4,39,5

In [ ]:
lr = LinearRegression(featuresCol="features", labelCol="log Count", maxIter=100)

start_syn = time.time()
lr_syn_20_model = lr.fit(syn_20_training)
end_syn = time.time()

elapsed_time = end_syn - start_syn
print(f"Training Linear Regression on 20% synopsis took: {elapsed_time:.2f} sec.")
print("Linear Regression (Training on 20% Synopsis):")

lr_syn_20_predictions = lr_syn_20_model.transform(test_data).withColumn("Count prediction", round(expm1("prediction"), 1))
lr_syn_20_predictions.select("features", "prediction", "log Count", "Count prediction", "Crime Count").show(10)

Training Linear Regression on 20% synopsis took: 24.89 sec.
Linear Regression (Training on 20% Synopsis):
+--------------------+------------------+------------------+----------------+-----------+
|            features|        prediction|         log Count|Count prediction|Crime Count|
+--------------------+------------------+------------------+----------------+-----------+
|(54,[5,43,51,52,5...| 2.568265928488678|2.9444389791664403|            12.0|         18|
|(54,[5,29,51,52,5...|2.7856001829509154|1.9459101490553132|            15.2|          6|
|(54,[5,38,51,52,5...|  2.62959845264524|2.1972245773362196|            12.9|          8|
|(54,[10,28,51,52,...| 2.120360104494921|3.4965075614664802|             7.3|         32|
|(54,[0,39,51,52,5...|3.3165828426973647|3.6635616461296463|            26.6|         38|
|(54,[0,35,51,52,5...| 3.376738334983365| 3.044522437723423|            28.3|         20|
|(54,[5,42,51,52,5...| 2.472910629565945|2.4849066497880004|            10.9|       

# Regression Evaluation on Synopses

Evaluation of ml model's results on the different synopses

In [ ]:
print_metrics(gbt_syn_40_predictions, "GBT Regressor (40% synopsis)", "log Count")
print_metrics(dt_syn_40_predictions, "Decision Tree Regressor(40% synopsis)", "log Count")
print_metrics(lr_syn_40_predictions, "Linear Regression (40% synopsis)", "log Count")

Evaluation of GBT Regressor (40% synopsis): 
r2: 0.5318009155300916
rmse: 0.6307207763082211
mae: 0.5101982394828317

Evaluation of Decision Tree Regressor(40% synopsis): 
r2: 0.4652730224901883
rmse: 0.6740434684555457
mae: 0.5401989970746696

Evaluation of Linear Regression (40% synopsis): 
r2: 0.6146340925628826
rmse: 0.5722140306835233
mae: 0.45345063897218



In [ ]:
print_metrics(gbt_syn_30_predictions, "GBT Regressor (30% synopsis)", "log Count")
print_metrics(dt_syn_30_predictions, "Decision Tree Regressor(30% synopsis)", "log Count")
print_metrics(lr_syn_30_predictions, "Linear Regression (30% synopsis)", "log Count")

Evaluation of GBT Regressor (30% synopsis): 
r2: 0.417886581011485
rmse: 0.7032757823022628
mae: 0.5760709237194195

Evaluation of Decision Tree Regressor(30% synopsis): 
r2: 0.3335975706812264
rmse: 0.7524716512218687
mae: 0.612817219702249

Evaluation of Linear Regression (30% synopsis): 
r2: 0.630653085638322
rmse: 0.5601948276981686
mae: 0.4555963230535096



In [ ]:
print_metrics(gbt_syn_20_predictions, "GBT Regressor (20% synopsis)", "log Count")
print_metrics(dt_syn_20_predictions, "Decision Tree Regressor(20% synopsis)", "log Count")
print_metrics(lr_syn_20_predictions, "Linear Regression (20% synopsis)", "log Count")

Evaluation of GBT Regressor (20% synopsis): 
r2: 0.13871727566389114
rmse: 0.8554501862083251
mae: 0.702682562948491

Evaluation of Decision Tree Regressor(20% synopsis): 
r2: 0.0669130241411392
rmse: 0.8903954270278517
mae: 0.730604264707293

Evaluation of Linear Regression (20% synopsis): 
r2: 0.47517888958205323
rmse: 0.6677709237530204
mae: 0.549997858894374



# Stop the Spark Session

In [ ]:
spark.stop()